In [ ]:
from feature_extraction import (run_extraction, extract_all_features, extract_features, feature_names,
                                mfcc, spectral_flatness, spectral_contrast, octave_bands)
from model import SVM, KNN, save_model, load_model, Standardized
from config import (KERNEL, C, GAMMA, K, SEED, DURATIONS, FOLDER_PATH, TEST_SNR, TEST_COMPRESSION, DOWN_SR,
                    SAMPLE_RATE, N_FFT, HOP_LENGTH, WINDOW_LENGTH, N_MFCC, MANIFEST_PATH, PROJECT_ROOT)
from evaluation import (grid_search, search_best_duration, evaluate_models, plot_robustness, show_table, show_confusion_grid)
from preprocessing import (prepare_data, load_dataset, split_data, SPLITS, align_onset_all,
                           load_data, align_to_onset, summarize_feature)
import numpy as np
from tqdm import tqdm
import pandas as pd

from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Audio

In [ ]:
original_dataset = dict(zip(SPLITS, split_data(load_dataset(FOLDER_PATH))))
original_dataset["train"].signals = align_onset_all(original_dataset["train"].signals)
original_features = extract_all_features(original_dataset["train"], desc="Extract Original Train Data")

## Feature extraction, step by step

One training clip followed through the whole pipeline, first as the original recording and then as its augmented copies:

1. **Waveform**, after onset alignment (`align_to_onset`)
2. **MFCC**, 13 coefficients per frame (`mfcc`)
3. **Spectral contrast**, one row per octave band (`spectral_contrast`)
4. **Spectral flatness** over the whole spectrum (`spectral_flatness`)
5. **Summarized version**: the mean and std of each feature over time (`summarize_feature`), which make up the 44-value vector the models get (`extract_features`)

The augmented copies are the files `prepare_data` wrote for this clip (listed in the manifest). For each augmentation the copy with the strongest degradation is used: the lowest SNR, the worst MP3 quality and the lowest sample rate.

In [ ]:
# Pick one training clip (already onset-aligned in the cell above)
SAMPLE_LABEL = "daluo"   # one of LABEL
sample_idx = original_dataset["train"].labels.index(SAMPLE_LABEL)
sample_name = original_dataset["train"].names[sample_idx]

# Its augmented training copies, from the manifest written by prepare_data
manifest = pd.read_csv(MANIFEST_PATH)
copies = manifest[(manifest["source"] == sample_name) & (manifest["split"] == "train")
                  & (manifest["augmentation"] != "original")].astype({"param": int})

# Strongest degradation of each kind: lowest SNR, worst MP3 quality (9 = worst), lowest sample rate
strongest = {"noise": ("min", "dB SNR"), "mp3": ("max", "VBR quality"), "downsample": ("min", "Hz")}

signals = {"original": original_dataset["train"].signals[sample_idx]}
for aug, (pick, unit) in strongest.items():
    rows = copies[copies["augmentation"] == aug]
    row = rows.loc[rows["param"].idxmin() if pick == "min" else rows["param"].idxmax()]
    # Same loading and alignment as the original clip: load_data -> align_to_onset
    signals[f"{aug} ({row.param} {unit})"] = align_to_onset(load_data(PROJECT_ROOT / row.path))
augmented_versions = [v for v in signals if v != "original"]

# Frame-wise features of every version, with the same settings as extract_features
features = {}
for version, y in signals.items():
    flatness, _ = spectral_flatness(y, sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH)
    features[version] = {
        "mfcc": mfcc(y, sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
                     win_length=WINDOW_LENGTH, n_mfcc=N_MFCC),
        "contrast": spectral_contrast(y, sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH),
        "flatness": flatness,
        "vector": extract_features(y),
    }

band_edges = octave_bands(SAMPLE_RATE)
band_labels = [f"{lo:.0f}-{hi:.0f} Hz" for lo, hi in zip(band_edges[:-1], band_edges[1:])]

# Shared colour scales, so the original and augmented plots can be compared directly
mfcc_range = (min(f["mfcc"].min() for f in features.values()), max(f["mfcc"].max() for f in features.values()))
contrast_range = (min(f["contrast"].min() for f in features.values()), max(f["contrast"].max() for f in features.values()))

print(f"Sample: {sample_name} ({SAMPLE_LABEL})")
for version, y in signals.items():
    print(f"  {version:<26} {len(y) / SAMPLE_RATE:.2f} s, {features[version]['mfcc'].shape[1]} frames")

### Original clip

#### 1. Waveform

In [ ]:
y = signals["original"]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(np.arange(len(y)) / SAMPLE_RATE, y, lw=0.6)
ax.set(title=f"Original waveform: {sample_name}", xlabel="Time (s)", ylabel="Amplitude")
ax.margins(x=0)
plt.show()
display(Audio(y, rate=SAMPLE_RATE))

#### 2. MFCC

All 13 coefficients per frame. Coefficient 0 (bottom row) follows the overall loudness of the frame, so it is far more negative than the others.

In [ ]:
M = features["original"]["mfcc"]
fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
im = ax.imshow(M, aspect="auto", origin="lower", cmap="coolwarm", vmin=mfcc_range[0], vmax=mfcc_range[1],
               extent=[0, M.shape[1] * HOP_LENGTH / SAMPLE_RATE, -0.5, N_MFCC - 0.5])
ax.set(title="Original MFCC", xlabel="Time (s)", ylabel="Coefficient", yticks=range(N_MFCC))
fig.colorbar(im, ax=ax, label="Value")
plt.show()

#### 3. Spectral contrast

The dB gap between the loudest and the quietest frequencies of each octave band. High means clear peaks (tonal), low means energy spread evenly (noise-like).

In [ ]:
S = features["original"]["contrast"]
fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
im = ax.imshow(S, aspect="auto", origin="lower", cmap="viridis", vmin=contrast_range[0], vmax=contrast_range[1],
               extent=[0, S.shape[1] * HOP_LENGTH / SAMPLE_RATE, -0.5, len(band_labels) - 0.5])
ax.set(title="Original spectral contrast", xlabel="Time (s)", yticks=range(len(band_labels)), yticklabels=band_labels)
fig.colorbar(im, ax=ax, label="Contrast (dB)")
plt.show()

#### 4. Spectral flatness

Geometric over arithmetic mean of the power spectrum: close to 1 for noise, close to 0 for a tone. Frames with too little energy are NaN and left out of the summary (gaps in the line). The y-axis is logarithmic.

In [ ]:
F = features["original"]["flatness"][0]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(np.arange(len(F)) * HOP_LENGTH / SAMPLE_RATE, F)
ax.set(title="Original spectral flatness", xlabel="Time (s)", ylabel="Flatness", yscale="log")
ax.margins(x=0)
plt.show()

#### 5. Summarized version

Each feature is reduced to its mean and std over the frames (`summarize_feature`), so every clip gives the same 44 values whatever its length. Bars are the means, error bars the std; the flatness axis is logarithmic.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [N_MFCC, len(band_labels), 2]})
for ax, name, ticks in zip(axes, ["mfcc", "contrast", "flatness"],
                           [[str(i) for i in range(N_MFCC)], band_labels, ["whole spectrum"]]):
    mean, std = summarize_feature(features["original"][name])
    ax.bar(range(len(mean)), mean, yerr=std, capsize=3)
    ax.set_xticks(range(len(mean)), ticks, rotation=45 if name == "contrast" else 0, ha="right" if name == "contrast" else "center")
    ax.set_title(f"{name}: mean ± std")
axes[2].set_yscale("log")   # flatness values are tiny; the error bar below 0 is cut off on a log axis
axes[0].set_xlabel("Coefficient")
fig.suptitle("Original: summarized features")
fig.tight_layout()
plt.show()

# The 44-value vector the models are trained on (extract_features), as feature x (mean, std)
# Column names look like "mfcc_mean_0" -> (feature, index, stat)
index = pd.MultiIndex.from_tuples([(f, int(i), s) for f, s, i in (n.rsplit("_", 2) for n in feature_names())],
                                  names=["feature", "index", "stat"])
summary = pd.Series(features["original"]["vector"], index=index).unstack("stat")
display(summary.loc[["mfcc", "flatness", "contrast"]])

#### 6. Combined feature vector

The six summaries are joined into one array, in the order `extract_features` uses. This 44-value array is one row of the model input `X`.

In [ ]:
mfcc_mean, mfcc_std = summarize_feature(features["original"]["mfcc"])
flat_mean, flat_std = summarize_feature(features["original"]["flatness"])
contrast_mean, contrast_std = summarize_feature(features["original"]["contrast"])

parts = {"mfcc mean": mfcc_mean, "mfcc std": mfcc_std,
         "flatness mean": flat_mean, "flatness std": flat_std,
         "contrast mean": contrast_mean, "contrast std": contrast_std}
combined = np.concatenate(list(parts.values()))

with np.printoptions(formatter={"float": "{:.4g}".format}, linewidth=120):
    start = 0
    for name, part in parts.items():
        print(f"[{start:2d}:{start + len(part):2d}]  {name:<14} {part}")
        start += len(part)

    print(f"\nCombined feature vector, shape {combined.shape}:")
    print(combined)

print("\nSame as extract_features(signal):", np.allclose(combined, features["original"]["vector"]))

### Augmented copies

The same steps for the augmented copies of the clip. Colour scales are shared with the original plots above.

#### 1. Waveform

At this scale the copies look almost the same as the original, so the red line shows what the augmentation changed (copy minus original).

In [ ]:
fig, axes = plt.subplots(len(augmented_versions), 1, figsize=(12, 2.6 * len(augmented_versions)), sharex=True, sharey=True)
for ax, version in zip(axes, augmented_versions):
    y_orig, y = signals["original"], signals[version]
    n = min(len(y), len(y_orig))   # both are onset-aligned, so they line up sample by sample
    ax.plot(np.arange(len(y)) / SAMPLE_RATE, y, lw=0.6, label=version)
    ax.plot(np.arange(n) / SAMPLE_RATE, y[:n] - y_orig[:n], lw=0.6, color="tab:red", label="copy - original")
    ax.set(title=version, ylabel="Amplitude")
    ax.legend(loc="upper right")
    ax.margins(x=0)
axes[-1].set_xlabel("Time (s)")
fig.tight_layout()
plt.show()

for version in augmented_versions:
    display(Markdown(f"**{version}**"), Audio(signals[version], rate=SAMPLE_RATE))

#### 2. MFCC

In [ ]:
fig, axes = plt.subplots(1, len(augmented_versions), figsize=(16, 4), sharey=True, layout="constrained")
for ax, version in zip(axes, augmented_versions):
    M = features[version]["mfcc"]
    im = ax.imshow(M, aspect="auto", origin="lower", cmap="coolwarm", vmin=mfcc_range[0], vmax=mfcc_range[1],
                   extent=[0, M.shape[1] * HOP_LENGTH / SAMPLE_RATE, -0.5, N_MFCC - 0.5])
    ax.set(title=version, xlabel="Time (s)")
axes[0].set(ylabel="Coefficient", yticks=range(N_MFCC))
fig.colorbar(im, ax=axes, label="Value")
fig.suptitle("Augmented MFCC")
plt.show()

#### 3. Spectral contrast

In [ ]:
fig, axes = plt.subplots(1, len(augmented_versions), figsize=(16, 4), sharey=True, layout="constrained")
for ax, version in zip(axes, augmented_versions):
    S = features[version]["contrast"]
    im = ax.imshow(S, aspect="auto", origin="lower", cmap="viridis", vmin=contrast_range[0], vmax=contrast_range[1],
                   extent=[0, S.shape[1] * HOP_LENGTH / SAMPLE_RATE, -0.5, len(band_labels) - 0.5])
    ax.set(title=version, xlabel="Time (s)")
axes[0].set(yticks=range(len(band_labels)), yticklabels=band_labels)
fig.colorbar(im, ax=axes, label="Contrast (dB)")
fig.suptitle("Augmented spectral contrast")
plt.show()

#### 4. Spectral flatness

All versions on one plot, the original in grey.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
for version in signals:
    F = features[version]["flatness"][0]
    ax.plot(np.arange(len(F)) * HOP_LENGTH / SAMPLE_RATE, F, label=version,
            **({"color": "0.6", "lw": 2} if version == "original" else {}))
ax.set(title="Spectral flatness: original vs augmented", xlabel="Time (s)", ylabel="Flatness", yscale="log")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
ax.margins(x=0)
plt.show()

#### 5. Summarized version

Means (bars) and std (error bars) of every version side by side, then the full 44-value vectors. The flatness axis is logarithmic.

In [ ]:
versions = list(signals)
width = 0.8 / len(versions)
fig, axes = plt.subplots(3, 1, figsize=(15, 11))
for ax, name, ticks in zip(axes, ["mfcc", "contrast", "flatness"],
                           [[str(i) for i in range(N_MFCC)], band_labels, ["whole spectrum"]]):
    for k, version in enumerate(versions):
        mean, std = summarize_feature(features[version][name])
        ax.bar(np.arange(len(mean)) + (k - (len(versions) - 1) / 2) * width, mean, width, yerr=std, capsize=2,
               label=version, **({"color": "0.6"} if version == "original" else {}))
    ax.set_xticks(range(len(ticks)), ticks)
    ax.set_title(f"{name}: mean ± std")
axes[2].set_yscale("log")   # flatness differs by orders of magnitude between versions
axes[0].legend(ncol=len(versions))
fig.suptitle("Summarized features: original vs augmented")
fig.tight_layout()
plt.show()

# Full feature vectors, one column per version
vectors = pd.DataFrame({version: features[version]["vector"] for version in versions}, index=feature_names())
display(vectors)

In [ ]:
SVM_params, _ = grid_search(
    {"kernel": KERNEL, "C": C, "gamma": GAMMA},
    SVM,
    X= original_features.X, y= original_features.labels, 
    excludes=[(("kernel", "gamma"), ("linear", "scale"))],  # gamma has no effect on the linear kernel
    random_state=SEED
)

In [ ]:
KNN_params, _ = grid_search(
    {"k": K},
    KNN,
    X= original_features.X, y= original_features.labels
)

In [ ]:
# svm_best_duration, svm_durations = search_best_duration(DURATIONS, Standardized(model=SVM(random_state=SEED, **SVM_params)), original_dataset["train"])

In [ ]:
# knn_best_duration, knn_durations = search_best_duration(DURATIONS, Standardized(model=KNN(**KNN_params)), original_dataset["train"])

In [ ]:
# print (svm_durations)
# print (knn_durations)

# accuracy_mean = {}
# best_accuracy = 0
# for key in svm_durations.keys():
#     accuracy_mean[key] = (svm_durations[key] + knn_durations [key])/2

# print(accuracy_mean)

# best_duration, best_accuracy = max(accuracy_mean.items(), key=lambda item: item[1])

# print(f"The best overall duration is {best_duration}s, with accuracy mean of {best_accuracy}")

In [ ]:
train, test = prepare_data()
data = {
    "train":train,
    "test": test,
}

# original, noise, mp3, downsample
original, noise, mp3, downsample = run_extraction(False, data)

testing = {
    "original": original["test"],
    "noise": noise["test"],
    "mp3": mp3["test"],
    "downsample": downsample["test"]
}

scenarios = {
    "original":              [original["train"]],
    "noise":                 [noise["train"]],
    "mp3":                   [mp3["train"]],
    "downsample":            [downsample["train"]],
    "original+noise":        [original["train"], noise["train"]],
    "original+mp3":          [original["train"], mp3["train"]],
    "original+noise+mp3":    [original["train"], noise["train"], mp3["train"]],
    "all":                   [original["train"], noise["train"], mp3["train"], downsample["train"]],
}

In [ ]:
svm_models = []
for scenario, data in tqdm(scenarios.items(), "Train SVM model for each scenario"):
    X_train = np.concatenate([part.X for part in data])
    y_train = np.concatenate([part.labels for part in data])

    svm_model = Standardized(SVM(random_state=SEED, **SVM_params)).fit(X_train, y_train)
    save_model(svm_model, (scenario+"_svm"))
    svm_models.append(svm_model)

In [ ]:
knn_models = []
for scenario, data in tqdm(scenarios.items(), "Train KNN model for each scenario"):
    X_train = np.concatenate([part.X for part in data])
    y_train = np.concatenate([part.labels for part in data])

    knn_model = Standardized(KNN(**KNN_params)).fit(X_train, y_train)
    save_model(knn_model, (scenario+"_knn"))
    knn_models.append(knn_model)

In [ ]:
svm_models = {scenario: load_model(scenario + "_svm") for scenario in scenarios}
knn_models = {scenario: load_model(scenario + "_knn") for scenario in scenarios}

# Evaluate every model on every test set and level
svm_results, svm_matrices = evaluate_models(svm_models, testing, "SVM")
knn_results, knn_matrices = evaluate_models(knn_models, testing, "KNN")
results = pd.concat([svm_results, knn_results], ignore_index=True)
matrices = {**svm_matrices, **knn_matrices}

# Store
results_dir = Path("pipeline_output/results")
results_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(results_dir / "results.csv", index=False)
pd.to_pickle(matrices, results_dir / "confusion_matrices.pkl")

for model_name in ("SVM", "KNN"):
    display(Markdown(f"## {model_name}"))
    show_table(model_name, "macro_f1", results)
    show_table(model_name, "accuracy", results)

    plot_robustness(results, model_name, "macro_f1")
    plot_robustness(results, model_name, "accuracy")
    plt.show()
    
    show_confusion_grid(matrices, model_name, "original")       # clean test clips
    
    for snr in TEST_SNR:
        show_confusion_grid(matrices, model_name, "noise", snr)       # noise test clips
    for q in TEST_COMPRESSION:
        show_confusion_grid(matrices, model_name, "mp3", q) # mp3 test clips
    for sr in DOWN_SR:
        show_confusion_grid(matrices, model_name, "downsample", sr) # downsample test clips